# seq2tags: MACCROBAT (convertido p/ schema do grafo) -> embeddings PubMedBERT -> classificador de nós

Pipeline com a estrutura nova do projeto (`diplo_graph03/`):

| Etapa | Módulo |
|---|---|
| 0. Converte o MACCROBAT original para o schema do grafo (`DISEASE`, `SYMPTOM`, `DRUG`, `EXAM`, `ANATOMY`, `EFFECT`, `RESULT`, `PATIENT`) | `preprocessing/dataset.py` (`convert_folder`) |
| 1. Lê os pares .txt/.ann já convertidos + divide treino/val/teste por caso | `preprocessing/dataset.py` (`load_docs`, `get_splits`) |
| 2. Embeddings PubMedBERT por subtoken (cache em `.npz`) | `modeling/seq2tags/features.py` (`BertEmbedder`, `build_token_dataset`) |
| 3. Classificador de nós (LogReg + StandardScaler) | `modeling/seq2tags/node_clasifier.py` (`NodeClassifier`) |
| 4. Avaliação por entidade (P/R/F1) — compartilhada entre as 3 abordagens | `evaluation.py` (`entity_f1`, `print_report`) |

O passo 0 só precisa rodar **uma vez**: ele grava uma cópia do MACCROBAT já com os tipos do
grafo em `MACCROBAT_GRAFO_DIR`. Se a pasta já existir e tiver arquivos, o notebook pula essa
conversão.

In [15]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()          # este notebook fica em notebooks/, ROOT = project03/
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from diplo_graph03 import config, schema
from diplo_graph03.preprocessing import dataset
from diplo_graph03.modeling.seq2tags import features
from diplo_graph03.modeling.seq2tags.node_clasifier import NodeClassifier, fix_bio, bio_to_entities
from diplo_graph03.modeling.seq2tags.edge_clasifier import (
    EdgeClassifier, build_graph_for_doc, type_pairs_from_meta,
)

print("MACCROBAT_DIR      :", config.MACCROBAT_DIR)
print("MACCROBAT_GRAFO_DIR:", config.MACCROBAT_GRAFO_DIR)
print("SPLITS_JSON        :", config.SPLITS_JSON)
print("EMBEDDINGS_DIR     :", config.EMBEDDINGS_DIR)

MACCROBAT_DIR      : C:\Users\hidal\Documents\GitHub\Diplotokens-P1\project03\data\external\MACCROBAT2020
MACCROBAT_GRAFO_DIR: C:\Users\hidal\Documents\GitHub\Diplotokens-P1\project03\data\interim\maccrobat_grafo
SPLITS_JSON        : C:\Users\hidal\Documents\GitHub\Diplotokens-P1\project03\configs\splits.json
EMBEDDINGS_DIR     : C:\Users\hidal\Documents\GitHub\Diplotokens-P1\project03\data\processed\embeddings


## 0. Converte o MACCROBAT original para o schema do grafo (roda só na 1ª vez)

In [16]:
if config.MACCROBAT_GRAFO_DIR.exists() and any(config.MACCROBAT_GRAFO_DIR.rglob("*.ann")):
    print("já convertido em", config.MACCROBAT_GRAFO_DIR, "- pulando convert_folder")
else:
    stats = dataset.convert_folder(config.MACCROBAT_DIR, config.MACCROBAT_GRAFO_DIR)
    for k in sorted(stats):
        print(f"{stats[k]:6d}  {k}")

já convertido em C:\Users\hidal\Documents\GitHub\Diplotokens-P1\project03\data\interim\maccrobat_grafo - pulando convert_folder


## 1. Leitura dos casos convertidos e divisão treino/val/teste

Os `Doc.entities` aqui já vêm com os 7 tipos de `schema.NODE_TYPES` (+ `PATIENT`) — bem menos
dispersos que os ~40 rótulos originais do MACCROBAT, o que deve ajudar bastante a precisão que
estava baixa com o `class_weight="balanced"`.

O `configs/splits.json` é **do time** (treino/teste fixos, usados pelas 3 abordagens) — não
regeramos ele aqui. Se ele ainda não tiver uma chave `"val"`, derivamos a validação a partir do
`"train"` oficial só neste notebook (com semente fixa, sem sobrescrever o arquivo do time).

In [17]:
docs = dataset.load_docs(config.MACCROBAT_GRAFO_DIR)
print(len(docs), "casos | entidades com posição desalinhada:", dataset.check_offsets(docs))

splits_time = dataset.load_splits(config.SPLITS_JSON)

if splits_time.get("val"):
    splits = splits_time
    print("splits.json do time já tem val:", {k: len(v) for k, v in splits.items()})
else:
    # separa validação a partir do treino oficial (ratios: 82% treino / 18% val / 0% "teste" descartado)
    treino_val = dataset.split_ids(splits_time["train"], ratios=(0.82, 0.18, 0.0), seed=42)
    splits = {"train": treino_val["train"], "val": treino_val["val"], "test": splits_time["test"]}
    print("val derivado do treino oficial (teste do time preservado):",
          {k: len(v) for k, v in splits.items()})

# sanity check: todo id do splits.json existe entre os casos carregados?
ids_docs = {d.id for d in docs}
ids_splits = {i for v in splits.values() for i in v}
print("ids do splits.json ausentes nos docs carregados:", sorted(ids_splits - ids_docs))
print("docs carregados que não estão em nenhum split    :", len(ids_docs - ids_splits))

200 casos | entidades com posição desalinhada: 0
splits.json do time já tem val: {'train': 140, 'val': 30, 'test': 30}
ids do splits.json ausentes nos docs carregados: []
docs carregados que não estão em nenhum split    : 0


### Sanity check: os tipos de entidade batem com o schema?

Se aparecer algo fora de `schema.GRAPH_NODE_TYPES`, é sinal de que o `convert_folder` não
rodou com a versão mais recente do `dataset.py`, ou que a pasta convertida está desatualizada
(nesse caso, apague `MACCROBAT_GRAFO_DIR` e rode a célula do passo 0 de novo).

In [18]:
tipos_encontrados = {e.label for d in docs for e in d.entities.values()}
fora_do_schema = tipos_encontrados - set(schema.GRAPH_NODE_TYPES)
print("tipos encontrados:", sorted(tipos_encontrados))
print("fora do schema (deveria ser vazio):", fora_do_schema)

tipos encontrados: ['ANATOMY', 'DISEASE', 'DRUG', 'EFFECT', 'EXAM', 'PATIENT', 'RESULT', 'SYMPTOM']
fora do schema (deveria ser vazio): set()


## 2. Embeddings PubMedBERT (roda uma vez; depois vem do cache em disco)

In [19]:
embedder = features.BertEmbedder(mask_digits=False)
features.build_token_dataset(docs, splits, embedder, config.EMBEDDINGS_DIR)

train = features.load_token_dataset(config.EMBEDDINGS_DIR, "train", embedder.name)
val = features.load_token_dataset(config.EMBEDDINGS_DIR, "val", embedder.name)
test = features.load_token_dataset(config.EMBEDDINGS_DIR, "test", embedder.name)
print("train:", train["X"].shape, "| val:", val["X"].shape, "| test:", test["X"].shape)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10520.92it/s]
[transformers] BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[train] cache encontrado: train_BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext.npz
[val] cache encontrado: val_BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext.npz
[test] cache encontrado: test_BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext.npz
train: (82488, 768) | val: (15947, 768) | test: (16362, 768)


### Sanity check: distribuição das tags BIO (treino)

Com o mapeamento para o schema do grafo, a razão entre o tipo mais frequente e o menos
frequente deve ter caído bastante em relação aos ~200x que víamos com os rótulos originais
do MACCROBAT — isso é o que deve consertar o `class_weight="balanced"`.

In [20]:
import numpy as np
from collections import Counter

tags_sem_prefixo = [t[2:] if t != "O" else t for t in train["y"]]
contagem = Counter(tags_sem_prefixo)
for tipo, n in contagem.most_common():
    print(f"{tipo:10s} {n:7d}")

O            56960
EXAM          7256
RESULT        4865
ANATOMY       4477
SYMPTOM       4346
DISEASE       2139
DRUG          1732
PATIENT        673
EFFECT          40


## 3. Classificador de nós

`NodeClassifier` normaliza o X (StandardScaler) e treina um LogisticRegression (`lbfgs`) —
ver `node_clasifier.py` para o porquê de não usar `saga` aqui (embeddings densos, diferente
do DictVectorizer esparso do P2).

In [21]:
clf = NodeClassifier(C=1.0, max_iter=200).fit(train["X"], train["y"])

pred_val = clf.predict(val["X"])
rep_val = clf.evaluate(val["y"], pred_val, val["doc"])
clf.print_report(rep_val)

tipo                              P      R     F1      n
ANATOMY                       0.309  0.544  0.394    421
DISEASE                       0.184  0.477  0.265    176
DRUG                          0.487  0.724  0.582    127
EFFECT                        0.000  0.000  0.000      0
EXAM                          0.461  0.685  0.551    682
PATIENT                       0.933  0.933  0.933     30
RESULT                        0.318  0.695  0.437    394
SYMPTOM                       0.281  0.540  0.369    476
micro                         0.340  0.621  0.439   2306


c:\Users\hidal\Documents\GitHub\Diplotokens-P1\project 1\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 200 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=200).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 4. Pós-processamento: BIO -> entidades (exemplo em 1 documento)

`fix_bio` corrige sequências inválidas; `bio_to_entities` converte as tags em entidades com
posição de caractere — é o formato que vai alimentar o classificador de arestas.

In [22]:
doc0 = val["doc"][0]
mask0 = val["doc"] == doc0

pred_val_fixed = fix_bio(list(pred_val))
tags0 = [t for t, m in zip(pred_val_fixed, mask0) if m]
offsets0 = val["offsets"][mask0]

ents0 = bio_to_entities(offsets0, tags0)
print(f"doc {doc0}: {len(ents0)} entidades previstas")
ents0[:10]

doc 24781756: 147 entidades previstas


[{'label': 'DISEASE',
  'char_start': 13,
  'char_end': 23,
  'token_start': 2,
  'token_end': 3},
 {'label': 'ANATOMY',
  'char_start': 41,
  'char_end': 47,
  'token_start': 9,
  'token_end': 10},
 {'label': 'RESULT',
  'char_start': 119,
  'char_end': 124,
  'token_start': 27,
  'token_end': 28},
 {'label': 'ANATOMY',
  'char_start': 134,
  'char_end': 145,
  'token_start': 32,
  'token_end': 34},
 {'label': 'ANATOMY',
  'char_start': 175,
  'char_end': 180,
  'token_start': 43,
  'token_end': 44},
 {'label': 'DISEASE',
  'char_start': 188,
  'char_end': 192,
  'token_start': 46,
  'token_end': 48},
 {'label': 'ANATOMY',
  'char_start': 263,
  'char_end': 269,
  'token_start': 59,
  'token_end': 60},
 {'label': 'ANATOMY',
  'char_start': 271,
  'char_end': 275,
  'token_start': 61,
  'token_end': 62},
 {'label': 'EXAM',
  'char_start': 276,
  'char_end': 282,
  'token_start': 62,
  'token_end': 63},
 {'label': 'EXAM',
  'char_start': 287,
  'char_end': 292,
  'token_start': 64,
  't

## 5. Avaliação final no teste (só rodar depois de fechar os hiperparâmetros na validação)

In [23]:
pred_test = clf.predict(test["X"])
rep_test = clf.evaluate(test["y"], pred_test, test["doc"])
clf.print_report(rep_test)

tipo                              P      R     F1      n
ANATOMY                       0.321  0.581  0.413    434
DISEASE                       0.203  0.596  0.303    166
DRUG                          0.515  0.814  0.631    129
EFFECT                        0.000  0.000  0.000      0
EXAM                          0.426  0.685  0.525    625
PATIENT                       0.935  0.967  0.951     30
RESULT                        0.308  0.724  0.433    337
SYMPTOM                       0.303  0.566  0.395    493
micro                         0.339  0.649  0.446   2214


## 6. Classificador de arestas

`features.build_edge_dataset` monta os pares de entidades (ouro, por enquanto — ver nota
abaixo) com `X = [v1 ; v2 ; |v1-v2| ; distância]` e `y` = tipo da relação ou `schema.NO_EDGE`.
Mesma ideia do classificador de nós: `neg_ratio` só no treino (subamostra os pares sem
relação), validação e teste ficam completos.

In [24]:
edges_train = features.build_edge_dataset(docs, train, neg_ratio=5)
edges_val = features.build_edge_dataset(docs, val)
edges_test = features.build_edge_dataset(docs, test)
print("treino:", edges_train["X"].shape, "| val:", edges_val["X"].shape, "| teste:", edges_test["X"].shape)
print("distribuição treino:", {lab: int((edges_train["y"] == lab).sum()) for lab in set(edges_train["y"])})

treino: (56130, 2305) | val: (38660, 2305) | teste: (33385, 2305)
distribuição treino: {np.str_('NO_EDGE'): 46775, np.str_('FAMILY_HISTORY'): 4, np.str_('DIAGNOSED_WITH'): 549, np.str_('OCCURS_IN'): 1152, np.str_('HAS_EFFECT'): 20, np.str_('UNDERGOES'): 2410, np.str_('TAKES'): 592, np.str_('AFFECTS'): 207, np.str_('PERSONAL_HISTORY'): 74, np.str_('REVEALS'): 2466, np.str_('HAS_SYMPTOM'): 1881}


In [25]:
edge_clf = EdgeClassifier(C=1.0, max_iter=200).fit(edges_train["X"], edges_train["y"])

pred_edges_val = edge_clf.predict(edges_val["X"])
rep_edges_val = edge_clf.evaluate(edges_val["y"], pred_edges_val)
edge_clf.print_report(rep_edges_val)

tipo                                P      R     F1      n
AFFECTS                         0.169  0.385  0.234     39
DIAGNOSED_WITH                  0.592  0.634  0.612    112
FAMILY_HISTORY                  0.000  0.000  0.000      1
HAS_SYMPTOM                     0.846  0.840  0.843    406
NO_EDGE                         0.996  0.963  0.979  36735
OCCURS_IN                       0.232  0.731  0.352    212
PERSONAL_HISTORY                0.474  0.692  0.562     13
REVEALS                         0.389  0.860  0.536    541
TAKES                           0.981  0.944  0.962    107
UNDERGOES                       0.902  0.947  0.924    494
micro (só arestas positivas)    0.522  0.844  0.645   1925


### Máscara de esquema (`predict_masked`)

O `predict()` acima escolhe a classe de maior probabilidade entre TODAS as 11 (10 tipos de
aresta + `NO_EDGE`), sem olhar os tipos dos 2 nós do par — por isso às vezes sai uma aresta
estruturalmente inválida (ex.: `UNDERGOES` entre `PATIENT -> DRUG`), que só aparece depois
como "problema estrutural" ao montar o `Graph`.

`predict_masked(X, type_pairs)` restringe, para cada par, as classes candidatas a
`schema.allowed_edge_types(tipo_src, tipo_dst) + [NO_EDGE]` antes de escolher a de maior
probabilidade — mesma ideia do `predict_edges_with_proba` do Projeto 2. `type_pairs_from_meta`
monta os `(tipo_src, tipo_dst)` a partir de `meta` + das entidades ouro do doc.

Comparamos abaixo P/R/F1 com e sem máscara: a métrica de validação só deve melhorar ou ficar
igual (nunca piora, já que a máscara nunca escolhe algo melhor do que o argmax livre dentro do
conjunto permitido) — o ganho real é não ter mais problemas estruturais no `Graph` final.

In [26]:
type_pairs_val = type_pairs_from_meta(docs, edges_val["meta"])
pred_edges_val_masked = edge_clf.predict_masked(edges_val["X"], type_pairs_val)
rep_edges_val_masked = edge_clf.evaluate(edges_val["y"], pred_edges_val_masked)

print("--- sem máscara ---")
edge_clf.print_report(rep_edges_val)
print("\n--- com máscara de esquema ---")
edge_clf.print_report(rep_edges_val_masked)

--- sem máscara ---
tipo                                P      R     F1      n
AFFECTS                         0.169  0.385  0.234     39
DIAGNOSED_WITH                  0.592  0.634  0.612    112
FAMILY_HISTORY                  0.000  0.000  0.000      1
HAS_SYMPTOM                     0.846  0.840  0.843    406
NO_EDGE                         0.996  0.963  0.979  36735
OCCURS_IN                       0.232  0.731  0.352    212
PERSONAL_HISTORY                0.474  0.692  0.562     13
REVEALS                         0.389  0.860  0.536    541
TAKES                           0.981  0.944  0.962    107
UNDERGOES                       0.902  0.947  0.924    494
micro (só arestas positivas)    0.522  0.844  0.645   1925

--- com máscara de esquema ---
tipo                                P      R     F1      n
AFFECTS                         0.457  0.410  0.432     39
DIAGNOSED_WITH                  0.946  0.777  0.853    112
FAMILY_HISTORY                  0.000  0.000  0.000      1
HAS_

### Montando o `Graph` previsto de 1 caso (sanity check)

`build_graph_for_doc` usa os nós OURO do caso (ainda não encadeamos com a saída do
`NodeClassifier` — ver nota no topo de `edge_clasifier.py`) + as arestas previstas (agora
mascaradas), e devolve um `schema.Graph`. `g.problems()` vazio e `g.edge_validity()` == (n, n)
confirmam que a saída está estruturalmente coerente com o esquema (todo `GraphEdge` liga os
tipos certos) — é exatamente isso que a máscara garante.

In [27]:
doc_exemplo = next(d for d in docs if str(d.id) == str(val["doc"][0]))
mask_doc = edges_val["meta"][:, 0] == doc_exemplo.id
g_pred = build_graph_for_doc(doc_exemplo, edges_val["meta"][mask_doc], pred_edges_val_masked[mask_doc])

print(f"doc {doc_exemplo.id}: {len(g_pred.nodes)} nós, {len(g_pred.edges)} arestas previstas")
print("problemas estruturais (deveria ser vazio):", g_pred.problems())
print("validade de esquema (ok, total):", g_pred.edge_validity())

# comparação com o grafo ouro do mesmo caso
g_ouro = schema.Graph.from_brat(doc_exemplo.entities, doc_exemplo.relations, doc_exemplo.id)
arestas_pred = {(g_pred.nodes[e.src].text, e.type, g_pred.nodes[e.dst].text) for e in g_pred.edges}
arestas_ouro = {(g_ouro.nodes[e.src].text, e.type, g_ouro.nodes[e.dst].text) for e in g_ouro.edges}
print("\nsó no previsto :", arestas_pred - arestas_ouro)
print("só no ouro     :", arestas_ouro - arestas_pred)
print("em comum       :", arestas_pred & arestas_ouro)

doc 24781756: 94 nós, 76 arestas previstas
problemas estruturais (deveria ser vazio): []
validade de esquema (ok, total): (76, 76)

só no previsto : {('Physical exam', np.str_('REVEALS'), '32\u2009cm'), ('tricuspid regurge', np.str_('OCCURS_IN'), 'cortical'), ('Apgar', np.str_('REVEALS'), '32\u2009cm'), ('Metabolic work up', np.str_('REVEALS'), '27\u2009mm'), ('head circumference', np.str_('REVEALS'), '2.3\u2009kg'), ('US', np.str_('REVEALS'), 'grew rapidly'), ('Birth weight', np.str_('REVEALS'), '9'), ('Metabolic work up', np.str_('REVEALS'), '277\u2009μm'), ('Physical exam', np.str_('REVEALS'), '9'), ('male', np.str_('PERSONAL_HISTORY'), 'febrile illness'), ('dilatation', np.str_('OCCURS_IN'), 'subcortical'), ('ammonia', np.str_('REVEALS'), 'normal'), ('head circumference', np.str_('REVEALS'), '9'), ('Physical exam', np.str_('REVEALS'), '27\u2009mm'), ('dilatation', np.str_('OCCURS_IN'), 'cortical'), ('alanine', np.str_('REVEALS'), 'normal'), ('cardiomegaly', np.str_('OCCURS_IN'), 'C

## 7. Avaliação final das arestas no teste (só rodar depois de fechar hiperparâmetros)

In [28]:
type_pairs_test = type_pairs_from_meta(docs, edges_test["meta"])
pred_edges_test = edge_clf.predict_masked(edges_test["X"], type_pairs_test)
rep_edges_test = edge_clf.evaluate(edges_test["y"], pred_edges_test)
edge_clf.print_report(rep_edges_test)

tipo                                P      R     F1      n
AFFECTS                         0.379  0.611  0.468     18
DIAGNOSED_WITH                  0.957  0.937  0.947     95
HAS_SYMPTOM                     0.978  0.978  0.978    408
NO_EDGE                         0.995  0.977  0.986  31580
OCCURS_IN                       0.428  0.759  0.547    249
PERSONAL_HISTORY                0.333  0.500  0.400      4
REVEALS                         0.484  0.830  0.612    487
TAKES                           1.000  1.000  1.000     96
UNDERGOES                       0.998  0.993  0.996    448
micro (só arestas positivas)    0.695  0.906  0.786   1805
